# Phase 5: Six-pack (Čech) — CPU 로컬 버전

Čech complex 기반 six-pack barcode를 **CPU(NumPy)**로 계산합니다.
(원본 Colab 노트북의 CuPy GPU 코드를 NumPy로 변환)

### Pipeline
1. 환경 설정
2. Čech complex 구축 (NumPy)
3. Six-pack barcode 계산 (image / kernel / cokernel / relative)
4. PI 벡터화 → 배치 계산 → 분류 평가

## 1. 환경 설정

In [ ]:
# 필요한 패키지 설치 (최초 1회)
!pip install gudhi persim scikit-learn psutil

In [ ]:
import os, time, gc, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from gudhi import SimplexTree
from collections import defaultdict
from persim import PersistenceImager
import persim.images_weights as weights

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.base import clone
import psutil
import warnings
warnings.filterwarnings('ignore')

print(f'NumPy version: {np.__version__}')
print('Running on CPU (no GPU required)')
print('All imports loaded.')

## 2. Čech Complex 구축 (CPU / NumPy)

원본의 CuPy 코드를 NumPy로 변환. 로직은 동일합니다.

In [ ]:
def compute_Cech_cpu(points, max_radius=5.0):
    """CPU (NumPy) Čech complex construction.
    거리 행렬 + 삼각형 외접원 계산을 NumPy 벡터화로 수행."""
    n = len(points)
    st = SimplexTree()

    # ── 0-simplices ──────────────────────────────────────────────
    for i in range(n):
        st.insert([i], filtration=0.0)

    # ── 1-simplices: 거리 행렬 ───────────────────────────────────
    pts = np.asarray(points, dtype=np.float64)
    diff = pts[:, None, :] - pts[None, :, :]   # (n, n, d)
    dist_mat = np.sqrt(np.sum(diff ** 2, axis=2))  # (n, n)
    del diff

    radius_mat = dist_mat / 2.0

    # valid edges: upper triangle, radius <= max_radius
    mask_edge = np.triu(radius_mat <= max_radius, k=1)
    edge_i, edge_j = np.where(mask_edge)
    edge_radii = radius_mat[edge_i, edge_j]

    valid_edges_set = set()
    for ei, ej, er in zip(edge_i, edge_j, edge_radii):
        st.insert([int(ei), int(ej)], filtration=float(er))
        valid_edges_set.add((int(ei), int(ej)))

    del mask_edge, edge_radii

    # ── 2-simplices: 삼각형 외접원 ────────────────────────────────
    n_edges = len(edge_i)

    if n_edges > 0:
        # 인접 리스트 구축
        adj = [set() for _ in range(n)]
        for ei, ej in zip(edge_i, edge_j):
            adj[ei].add(ej)
            adj[ej].add(ei)

        # 삼각형 열거: edge (i,j)에 대해 공통 이웃 k > j 찾기
        tri_list = []
        for ei, ej in zip(edge_i, edge_j):
            common = adj[ei].intersection(adj[ej])
            for k in common:
                if k > ej:  # 중복 제거: i < j < k
                    tri_list.append((int(ei), int(ej), int(k)))

        del adj

        if tri_list:
            triangles = np.array(tri_list, dtype=np.int32)
            del tri_list

            # 배치 외접원 계산 (NumPy)
            p0 = pts[triangles[:, 0]]  # (T, d)
            p1 = pts[triangles[:, 1]]  # (T, d)
            p2 = pts[triangles[:, 2]]  # (T, d)

            # 세 변의 길이
            a = np.linalg.norm(p1 - p2, axis=1)  # (T,)
            b = np.linalg.norm(p0 - p2, axis=1)
            c = np.linalg.norm(p0 - p1, axis=1)
            del p0, p1, p2

            # Heron's formula로 넓이 계산
            s = (a + b + c) / 2.0
            area_sq = s * (s - a) * (s - b) * (s - c)
            area_sq = np.maximum(area_sq, 0.0)  # 수치 오류 방지

            # 외접원 반지름: R = abc / (4 * area)
            degenerate = area_sq <= 1e-10
            safe_area = np.where(degenerate, np.ones_like(area_sq), area_sq)
            circumradius = np.where(
                degenerate,
                np.maximum(np.maximum(a, b), c) / 2.0,
                (a * b * c) / (4.0 * np.sqrt(safe_area))
            )
            del a, b, c, s, area_sq, safe_area, degenerate

            # max_radius 이하만 필터링
            valid_mask = circumradius <= max_radius
            valid_tri = triangles[valid_mask]
            valid_radii = circumradius[valid_mask]
            del circumradius, valid_mask

            # SimplexTree에 삽입
            for (ti, tj, tk), tr in zip(valid_tri, valid_radii):
                st.insert([int(ti), int(tj), int(tk)], filtration=float(tr))

            del valid_tri, valid_radii, triangles
        else:
            del tri_list

    del pts, dist_mat, radius_mat, edge_i, edge_j
    del valid_edges_set

    return st

## 3. Boundary Matrix & Column Reduction (CPU)

In [ ]:
def divide_filtration(st):
    simplex_filt_pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    list_simplex = [pair[0] for pair in simplex_filt_pairs]
    list_filt = [pair[1] for pair in simplex_filt_pairs]
    return list_simplex, list_filt


def _build_boundary(simplices):
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    boundary = []
    for s in simplices:
        if len(s) <= 1:
            boundary.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx:
                    rows.add(sf_to_idx[face])
            boundary.append(rows)
    return boundary


def _reduce_with_V(columns):
    m = len(columns)
    R = [set(col) for col in columns]
    V = [{i} for i in range(m)]
    low = [-1] * m
    pivot_of_row = {}
    for i in range(m):
        while R[i]:
            li = max(R[i])
            if li in pivot_of_row:
                owner = pivot_of_row[li]
                R[i] ^= R[owner]
                V[i] ^= V[owner]
            else:
                pivot_of_row[li] = i
                low[i] = li
                break
        else:
            low[i] = -1
    return R, low, V

## 4. Six-pack Barcode 계산 (Image / Kernel / Cokernel / Relative)

메모리 최적화: 중간 행렬을 즉시 해제합니다.

In [ ]:
def compute_all_barcodes(A, B, max_radius=5):
    """Compute Image, Kernel, Cokernel, Relative barcodes for L ⊆ K.
    L = Cech(A), K = Cech(A ∪ B). CPU NumPy version."""
    total = np.concatenate([A, B], axis=0)
    a = len(A)

    st = compute_Cech_cpu(total, max_radius=max_radius)
    del total

    simplices, filt = divide_filtration(st)
    del st
    gc.collect()

    m = len(simplices)

    # ── L / K\L 분류 ──────────────────────────────────────────────
    in_L        = [all(v < a for v in s) for s in simplices]
    idx_L       = [i for i, b in enumerate(in_L) if b]
    idx_KmL     = [i for i, b in enumerate(in_L) if not b]
    set_idx_KmL = set(idx_KmL)
    g2L         = {g: pos for pos, g in enumerate(idx_L)}

    # ── Step 1: Df, Dg 소거 ───────────────────────────────────────
    Df = _build_boundary(simplices)
    Rf, lowf, Vf = _reduce_with_V(Df)

    boundary_L = [{g2L[r] for r in Df[g_idx] if r in g2L} for g_idx in idx_L]
    Rg, lowg, Vg = _reduce_with_V(boundary_L)
    del boundary_L

    # ── Step 2: Dim 소거 → Rim ────────────────────────────────────
    row_order    = idx_L + idx_KmL
    row_remap    = {g: i for i, g in enumerate(row_order)}
    inv_row_remap = {i: g for g, i in row_remap.items()}
    del row_order

    Dim = [{row_remap[r] for r in Df[col_idx]} for col_idx in range(m)]
    Rim, lowim, _ = _reduce_with_V(Dim)
    del Dim, _
    gc.collect()

    Vim = [{row_remap[r] for r in Vf[col_idx]} for col_idx in range(m)]
    del Vf
    gc.collect()

    # ── Step 3: Dker 소거 ─────────────────────────────────────────
    cycle_cols = [i for i in range(m) if not Rim[i]]
    Dker       = [Vim[c] for c in cycle_cols]
    if Dker:
        _, lowker, _ = _reduce_with_V(Dker)
        del _
    else:
        lowker = []
    del Dker
    cycle_pos = {c: pos for pos, c in enumerate(cycle_cols)}
    del cycle_cols, Vim
    gc.collect()

    # ── Step 4: Dcok 소거 ─────────────────────────────────────────
    Dcok = []
    for i in range(m):
        if in_L[i]:
            jL = g2L[i]
            if not Rg[jL]:
                Dcok.append({idx_L[pos] for pos in Vg[jL]})
                continue
        Dcok.append(set(Df[i]))
    _, lowcok, _ = _reduce_with_V(Dcok)
    del Dcok, _, Vg
    gc.collect()

    # ── Step 5: Drel 소거 (H*(K,L)) ────────────────────────────────
    KmL_pos = {g: pos for pos, g in enumerate(idx_KmL)}
    Drel    = [{KmL_pos[r] for r in Df[i] if r in set_idx_KmL} for i in idx_KmL]
    del Df
    gc.collect()

    Rrel, lowrel, _ = _reduce_with_V(Drel)
    del Drel, _, lowrel, KmL_pos
    gc.collect()

    # ── Bar 추출 ─────────────────────────────────────────────────
    def _format(bars_dict):
        out = {}
        for p in [0, 1]:
            if p in bars_dict and bars_dict[p]:
                arr = np.array(bars_dict[p])
                out[p] = arr[np.lexsort((arr[:, 1], arr[:, 0]))]
            else:
                out[p] = np.empty((0, 2))
        return out

    # Image bars
    image_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        sigma = inv_row_remap[lowim[tau]]
        if sigma in g2L:
            b, d = filt[sigma], filt[tau]
            if b != d:
                p = len(simplices[sigma]) - 1
                image_bars[p].append((b, d))

    # Kernel bars
    kernel_bars = defaultdict(list)
    for tau in idx_L:
        jL = g2L[tau]
        if not Rg[jL] or Rf[tau] or tau not in cycle_pos:
            continue
        lc = cycle_pos[tau]
        if lc >= len(lowker):
            continue
        ll = lowker[lc]
        if ll == -1:
            continue
        sigma = inv_row_remap[ll]
        if in_L[sigma]:
            continue
        b, d = filt[sigma], filt[tau]
        if b != d:
            p = len(simplices[sigma]) - 2
            if p >= 0:
                kernel_bars[p].append((b, d))
    del lowker, cycle_pos

    # Cokernel bars
    cok_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        if inv_row_remap[lowim[tau]] not in set_idx_KmL:
            continue
        lc = lowcok[tau]
        if lc == -1:
            continue
        b, d = filt[lc], filt[tau]
        if b != d:
            p = len(simplices[lc]) - 1
            cok_bars[p].append((b, d))
    del lowcok

    # Relative bars (H*(K,L))
    rel_bars = defaultdict(list)
    for pos in range(len(idx_KmL)):
        if not Rrel[pos]:
            continue
        sigma_local = max(Rrel[pos])
        sigma = idx_KmL[sigma_local]
        tau   = idx_KmL[pos]
        b, d  = filt[sigma], filt[tau]
        if abs(b - d) > 1e-12:
            p = len(simplices[sigma]) - 1
            rel_bars[p].append((b, d))

    del Rf, lowf, Rg, lowg, Rim, lowim, Rrel
    del in_L, idx_L, idx_KmL, set_idx_KmL, g2L
    del row_remap, inv_row_remap, simplices, filt
    gc.collect()

    result = {
        'image'    : _format(image_bars),
        'kernel'   : _format(kernel_bars),
        'cokernel' : _format(cok_bars),
        'relative' : _format(rel_bars),
    }
    del image_bars, kernel_bars, cok_bars, rel_bars
    gc.collect()
    return result

## 5. Ordinary Persistence Barcode & PI 벡터화

In [ ]:
def compute_Persistence_barcode(A, max_radius=5):
    fil_A = compute_Cech_cpu(A, max_radius=max_radius)
    fil_A.persistence()
    bar_A = {}
    for dim in [0, 1]:
        bars = fil_A.persistence_intervals_in_dimension(dim)
        bars = [b for b in bars if b[1] != np.inf and b[1] - b[0] > 1e-5]
        bar_A[dim] = np.array(bars) if bars else np.empty((0, 2))
    del fil_A
    return bar_A


def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    for key in barcodes:
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))

    vector = {}

    # H0
    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 1)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}
    bars_h0 = np.array(barcodes[0])
    if len(bars_h0) > 0:
        img_h0 = pers_imager_h0.transform(bars_h0, skew=False)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))
    img0_1d = np.mean(img_h0, axis=0)
    del img_h0, pers_imager_h0

    # H1
    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}
    bars_h1 = np.array(barcodes[1])
    if len(bars_h1) > 0:
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))
    del pers_imager_h1

    if normalization:
        vector[0] = img0_1d / np.max(img0_1d) if np.max(img0_1d) > 0 else img0_1d
        vector[1] = img_h1.flatten() / np.max(img_h1) if np.max(img_h1) > 0 else img_h1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img_h1.flatten()

    del img_h1
    return vector

## 6. 배치 계산 (512 샘플)

In [ ]:
# ── 경로 설정 (로컬용) ─────────────────────────────────────────────
BASE_DIR = os.path.join(os.getcwd(), 'Data', 'ParamSweep_Input')
OUT_DIR = os.path.join(os.getcwd(), 'Data', 'Sixpack_Cech')
os.makedirs(OUT_DIR, exist_ok=True)

MAX_RADIUS = 5  # Čech max radius
A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]
print(f'Total samples: {len(PARAM_LIST)}')
print(f'Input dir: {BASE_DIR}')
print(f'Output dir: {OUT_DIR}')
print(f'Max Čech radius: {MAX_RADIUS}')
print(f'Input exists: {os.path.exists(BASE_DIR)}')

In [ ]:
def get_ram_usage_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def process_single_sample(idx, params, base_dir, out_dir, max_radius):
    """단일 샘플 처리 — 함수 스코프로 메모리 자동 해제."""
    folder = os.path.join(base_dir, f'ParamSweep_{idx}_Output')
    pos_file = os.path.join(folder, f'Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')
    types_file = os.path.join(folder, f'Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')

    types = np.loadtxt(types_file, dtype=int)
    positions = np.loadtxt(pos_file, delimiter=',')
    A = positions[types == 1]
    B = positions[types == 2]
    del types, positions

    # Six-pack barcodes (CPU Čech)
    six_pack_A2B = compute_all_barcodes(A, B, max_radius=max_radius)
    gc.collect()

    six_pack_B2A = compute_all_barcodes(B, A, max_radius=max_radius)
    gc.collect()

    # Ordinary persistence barcodes (CPU Čech)
    total_pts = np.concatenate([A, B])
    PB_total = compute_Persistence_barcode(total_pts, max_radius=max_radius)
    del total_pts

    PB_A = compute_Persistence_barcode(A, max_radius=max_radius)
    PB_B = compute_Persistence_barcode(B, max_radius=max_radius)
    del A, B

    six_pack_A2B.update({'complex': PB_total, 'sub_complex': PB_A})
    six_pack_B2A.update({'complex': PB_total, 'sub_complex': PB_B})
    del PB_total, PB_A, PB_B

    # PI vectorization
    PI_A2B = {}
    for key in list(six_pack_A2B.keys()):
        PI_A2B[key] = compute_PIs(six_pack_A2B[key], normalization=False)
        del six_pack_A2B[key]
    del six_pack_A2B

    PI_B2A = {}
    for key in list(six_pack_B2A.keys()):
        PI_B2A[key] = compute_PIs(six_pack_B2A[key], normalization=False)
        del six_pack_B2A[key]
    del six_pack_B2A
    gc.collect()

    save_path = os.path.join(out_dir, f'Sixpack_Cech_{idx}.npz')
    np.savez_compressed(save_path, PI_A2B, PI_B2A)
    del PI_A2B, PI_B2A
    gc.collect()
    return save_path

In [ ]:
# ⚠️ CPU 계산은 GPU보다 느립니다. 이미 계산된 파일은 건너뜁니다.

START_IDX = 1
END_IDX = 512

for idx in range(START_IDX, END_IDX + 1):
    save_path = os.path.join(OUT_DIR, f'Sixpack_Cech_{idx}.npz')
    if os.path.exists(save_path):
        continue

    ram_mb = get_ram_usage_mb()
    print(f'[{idx}/512] 시작 (RAM: {ram_mb:.0f}MB)')
    start_time = time.time()

    try:
        params = PARAM_LIST[idx - 1]
        saved = process_single_sample(idx, params, BASE_DIR, OUT_DIR, MAX_RADIUS)
        elapsed = time.time() - start_time
        ram_mb = get_ram_usage_mb()
        print(f'[{idx}/512] 완료 ({elapsed:.1f}초, RAM: {ram_mb:.0f}MB)')

    except Exception as e:
        print(f'[{idx}/512] 에러: {e}')

    gc.collect()

print('\n배치 계산 완료!')

## 7. Ground Truth & 평가 함수

In [ ]:
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1, M2, M3, M4, M5, M6, M7, M8])

def get_label_from_index(task_id):
    idx = task_id - 1
    return GROUND_TRUTH_M[idx % 64 // 8][idx // 64][idx % 8]

ADJACENT_PHASES = {
    0:[1,2], 1:[0,3,4], 2:[0,3,4], 3:[1,2,4],
    4:[1,2,3,5,8,11], 5:[4,6,7,11], 6:[5,7,9,10],
    7:[5,6,9,10], 8:[4,9,10], 9:[6,7,8,10],
    10:[6,7,8,9], 11:[4,5],
}

def soft_accuracy_score(y_true, y_pred, adjacent_dict=ADJACENT_PHASES):
    n = 0
    for t, p in zip(y_true, y_pred):
        if t == p or (t in adjacent_dict and p in adjacent_dict[t]):
            n += 1
    return n / len(y_true) if len(y_true) > 0 else 0.0

print(f'Classes: {np.unique(GROUND_TRUTH_M)} ({len(np.unique(GROUND_TRUTH_M))} classes)')

## 8. Sixpack_Cech 벡터 로드

In [ ]:
def load_sixpack_cech_data(data_dir):
    npz_files = sorted(glob.glob(os.path.join(data_dir, 'Sixpack_Cech_*.npz')))
    if not npz_files:
        return None, None, None
    X_list, y_list, indices = [], [], []
    for filepath in npz_files:
        filename = os.path.basename(filepath)
        try:
            sim_idx = int(filename.split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(filepath, allow_pickle=True)
            arr_0 = data['arr_0'].item() if hasattr(data['arr_0'], 'item') else data['arr_0']
            arr_1 = data['arr_1'].item() if hasattr(data['arr_1'], 'item') else data['arr_1']
            features = []
            for arr in [arr_0, arr_1]:
                if isinstance(arr, dict):
                    for key in sorted(arr.keys()):
                        val = arr[key]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()):
                                features.extend(np.array(val[dk]).flatten())
                        else:
                            features.extend(np.array(val).flatten())
                else:
                    features.extend(np.array(arr).flatten())
            X_list.append(features)
            y_list.append(label)
            indices.append(sim_idx)
        except Exception as e:
            print(f'  Error: {filename} - {e}')
    if not X_list:
        return None, None, None
    return np.nan_to_num(np.array(X_list)), np.array(y_list), indices

X, y, idx = load_sixpack_cech_data(OUT_DIR)
if X is not None:
    print(f'Loaded: X.shape={X.shape}, y.shape={y.shape}')
else:
    print('No data found. Run batch computation first.')

## 9. 분류 평가 (5-fold Stratified CV)

In [ ]:
def evaluate_classifiers(X, y, pca_dim=20, n_splits=5):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    if pca_dim and X_s.shape[1] > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=42)
        X_r = pca.fit_transform(X_s)
    else:
        X_r = X_s

    clfs = {
        'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
        'KNN (k=12)': KNeighborsClassifier(n_neighbors=12),
        'SVM (RBF, C=1)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
        'SVM (Linear, C=1)': SVC(kernel='linear', C=1.0, random_state=42),
        'SVM (RBF, C=0.5)': SVC(kernel='rbf', C=0.5, gamma='scale', random_state=42),
        'SVM (RBF, C=2)': SVC(kernel='rbf', C=2.0, gamma='scale', random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    }
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = {}
    for name, tmpl in clfs.items():
        soft_a, strict_a, f1_s = [], [], []
        for tr, te in skf.split(X_r, y):
            c = clone(tmpl)
            c.fit(X_r[tr], y[tr])
            yp = c.predict(X_r[te])
            soft_a.append(soft_accuracy_score(y[te], yp))
            strict_a.append(accuracy_score(y[te], yp))
            f1_s.append(f1_score(y[te], yp, average='weighted', zero_division=0))
        results[name] = {
            'soft': (np.mean(soft_a)*100, np.std(soft_a)*100),
            'strict': (np.mean(strict_a)*100, np.std(strict_a)*100),
            'f1': (np.mean(f1_s)*100, np.std(f1_s)*100),
        }
    return results

if X is not None:
    for pca_d in [20, 50, None]:
        label = f'PCA={pca_d}D' if pca_d else 'No PCA'
        print(f'\n{"="*70}')
        print(f'Sixpack_Cech (CPU) — {label}, 5-fold CV')
        print(f'{"="*70}')
        res = evaluate_classifiers(X, y, pca_dim=pca_d)
        for n, r in res.items():
            print(f'  {n:25s} | Soft: {r["soft"][0]:.2f}±{r["soft"][1]:.2f}%  '
                  f'| Strict: {r["strict"][0]:.2f}±{r["strict"][1]:.2f}%  '
                  f'| F1: {r["f1"][0]:.2f}±{r["f1"][1]:.2f}%')

## 완료

Phase 5 (CPU 로컬 버전) 실험 완료.